In [ ]:
import urllib.request
import gzip
import shutil
import os
import pybedtools

# --- STEP 1: Download the COMPLETE hg38 Genome ---
# This URL includes all alternative loci and decoys (chrKI, chrGL, etc.)
url = "http://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/analysisSet/hg38.analysisSet.fa.gz"
genome_gz = "hg38_complete.fa.gz"
genome_fa = "hg38_complete.fa"

if not os.path.exists(genome_fa):
    print(f"1. Downloading complete hg38 genome (~3.1 GB)...")
    urllib.request.urlretrieve(url, genome_gz)
    
    print("2. Unzipping...")
    with gzip.open(genome_gz, 'rb') as f_in:
        with open(genome_fa, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
    
    os.remove(genome_gz) # Clean up .gz file
    print(f"Success! '{genome_fa}' is ready.")
else:
    print(f"Genome file '{genome_fa}' already exists. Skipping download.")

# --- STEP 2: Load Your BED File ---
# REPLACE 'your_dyak1a_file.bed' with your actual filename
bed_filename = 'your_dyak1a_file.bed' 

if os.path.exists(bed_filename):
    print(f"\n3. Loading BED file: {bed_filename}")
    bed_file = pybedtools.BedTool(bed_filename)
    print(f"Loaded {len(bed_file)} peaks.")
    
    # Check for chrKI to confirm we need the full genome
    chroms = set([f.chrom for f in bed_file])
    ki_chroms = [c for c in chroms if 'KI' in c or 'GL' in c]
    if ki_chroms:
        print(f"Detected alternative contigs: {ki_chroms[:3]}... (Full genome required)")

    # --- STEP 3: Extract Sequences ---
    print("\n4. Extracting sequences (this may take 1-2 mins)...")
    # s=True respects strand; name=True uses peak names in FASTA
    sequences = bed_file.sequence(fi=genome_fa, s=True, name=True)
    
    print(f"Done! Output file: {sequences.seqfn}")
    print(f"Total sequences extracted: {len(sequences)}")
    
    # Verify no warnings occurred by checking a few lines
    with open(sequences.seqfn, 'r') as f:
        print("\nFirst 2 entries in your new FASTA file:")
        for i in range(4):
            print(f.readline().strip())
            
else:
    print(f"ERROR: Could not find '{bed_filename}'. Please check the filename.")   